In [0]:
class Silver_constructors():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        read_df=spark.read.table('formula1_race.bronze.constructors')
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
            colrename_df= (colrename_df.withColumnRenamed('name','constructor_team')
                                     .withColumnRenamed('nationality','constructor_nationality')
                           )
            
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col
        apply_tran_df= (colrename_df.selectExpr("constructor_id","constructor_ref","constructor_team","constructor_nationality","constructors_ingestion_date","source")
                        )
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.mode("overwrite").saveAsTable("formula1_race.silver.constructors")
        display(spark.sql("select count(*) from formula1_race.silver.constructors"))
        print("Data write into sliver constructors table is Done")
       
    
    def process(self):
        print("Started silver-ingestion-constructors  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)
        
    


In [0]:
Silver_constructors_instance = Silver_constructors("constructors")
Silver_constructors_instance.process()